# Food balances in calories and protein, from crops to food

!!! info ""
    :octicons-person-16: **[Pablo Rosado](https://ourworldindata.org/team/pablo-rosado)** • :octicons-calendar-16: November 13, 2026 *(last edit)* • [**:octicons-mail-16: Feedback**](mailto:info@ourworldindata.org?subject=Feedback%20on%20technical%20publication%20-%20Food%20balances%20in%20calories%20and%20protein%2C%20from%20crops%20to%20food)

## Introduction

FAOSTAT, the statistical database of the Food and Agriculture Organization of the United Nations (FAO), publishes food balances.
For each country and year, a food balance records how much of each food item was produced, imported, exported, stored, and used.
FAOSTAT gives almost all of these figures only in tonnes.
The exception is the food available to eat, which FAOSTAT also gives in calories and in protein.

In this technical publication we explain how we convert the other figures of the balance from tonnes into calories and protein.
The result follows the calories, protein, and weight of each country's food supply, from the crops harvested to the food available to eat.
On the way, food is traded, stored, sown as seed, lost, put to other uses, processed in factories, and fed to animals, which give some of it back as meat, milk, and eggs.

For example, in 2023 the world harvested crops with about 6,000 kilocalories (kcal) per person per day, and about 3,000 kcal per person per day were available as food.

<!-- TODO: The link to the article is a placeholder. Update it once the article is published. -->

This technical publication is a companion to the article [The world produces enough calories to feed a global population twice the size. Where do they go?](https://ourworldindata.org/the-world-produces-enough-calories-to-feed-a-global-population-twice-the-size-where-do-they-go), and in particular to its chart that follows each country's food from the crops it grows to the food available to eat.

This publication includes the code used to calculate the numbers quoted in the text.
You can run this notebook on [Google Colab](https://colab.research.google.com/github/owid/etl/blob/master/docs/analyses/food_supply_chain/food_supply_chain.ipynb), or [download the file](food_supply_chain.ipynb) to run it on your computer.

In [1]:
%%capture
# Whether you run this code in your local computer or on Google Colab, you first need to install the required dependencies:
# * owid-catalog: a library that lets you explore and load datasets from OWID.
# * plotly: a library that lets you create interactive visualizations.
%pip install --upgrade owid-catalog plotly

import contextlib
import io
import urllib.error
import urllib.request

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from owid.catalog import fetch
from plotly.subplots import make_subplots

CATALOG_URL = "https://catalog.ourworldindata.org"
# Datasets from our catalog with the data used in this notebook.
# Our chain built from FAOSTAT's Supply Utilization Accounts (SCL), created by the ETL step
# data://garden/faostat/2026-09-04/food_supply_chain_scl.
CHAIN_SCL_DATASET = "garden/faostat/2026-09-04/food_supply_chain_scl"
# Our chain built from FAOSTAT's Food Balance Sheets (FBS), created by the ETL step
# data://garden/faostat/2026-09-04/food_supply_chain_fbs.
CHAIN_FBS_DATASET = "garden/faostat/2026-09-04/food_supply_chain_fbs"
# FAOSTAT's Supply Utilization Accounts (SCL), on which the first chain is based.
FAOSTAT_SCL_DATASET = "garden/faostat/2026-02-25/faostat_scl"

# Regions in the tables of the chain, which we build from countries.
REGIONS = [
    "World",
    "Africa",
    "Asia",
    "Europe",
    "North America",
    "Oceania",
    "South America",
    "Low-income countries",
    "Lower-middle-income countries",
    "Upper-middle-income countries",
    "High-income countries",
    "European Union (27)",
]
# Stages of the chain, in order, with their names, and whether they add to the chain (1) or subtract from it (-1).
# The chain ends on the stage "food".
STAGES = {
    "crop_production": ("Crop production", 1),
    "imports": ("Imports", 1),
    "exports": ("Exports", -1),
    "stock_variation": ("Stock variation", -1),
    "seed": ("Seed", -1),
    "losses": ("Losses", -1),
    "other_uses": ("Other uses", -1),
    "processing_net": ("Processing, net", -1),
    "feed": ("Feed", -1),
    "animal_products": ("Animal products", 1),
    "tourist_consumption": ("Tourist consumption", -1),
    "data_adjustments": ("Data adjustments", -1),
}


def is_published(dataset):
    """Whether a dataset is published in our public catalog (where its index file exists)."""
    request = urllib.request.Request(f"{CATALOG_URL}/{dataset}/index.json", method="HEAD")
    try:
        urllib.request.urlopen(request, timeout=30)
    except urllib.error.HTTPError as error:
        if error.code == 404:
            return False
        raise
    return True


def load_table(dataset, table_name):
    """Load a table from our public catalog or, if its dataset is not published there yet, from a local ETL checkout.

    The table is returned as a plain dataframe: this notebook does not need its metadata.
    """
    if not is_published(dataset):
        from owid.catalog import Dataset

        from etl.paths import DATA_DIR

        tb = pd.DataFrame(Dataset(DATA_DIR / dataset).read(table_name))
    else:
        # NOTE: fetch() prints a progress spinner, which is silenced here.
        with contextlib.redirect_stdout(io.StringIO()):
            tb = pd.DataFrame(fetch(f"{dataset}/{table_name}").reset_index())
    return tb.astype({"country": str})


# Load the chain built from SCL, in kilocalories and in grams of protein per person per day.
energy = load_table(CHAIN_SCL_DATASET, "energy")
protein = load_table(CHAIN_SCL_DATASET, "protein")

## The stages of the chain

We call the result a chain: it starts at the crops harvested, each stage adds to it or subtracts from it, and it ends at the food available to eat.

| Stage | | Definition |
|---|---|---|
| Crop production | Starts the chain | Crops harvested in the country, before they are made into products such as flour or oils. |
| Imports | Added | Food products bought from other countries. |
| Exports | Subtracted | Food products sold to other countries. |
| Stock variation | Subtracted | Food products added to stocks, minus those taken out. |
| Seed | Subtracted | Crops set aside as seed for the next harvest. |
| Losses | Subtracted | Food lost in storage, transport, and processing, between the farm and the shop. |
| Other uses | Subtracted | Food products used for purposes other than food or feed, such as biofuels and soap. |
| Processing, net | Subtracted | Food products sent to be processed, for example milled, crushed, or brewed, minus the processed products made from them, such as flour, oils, sugar, and beer. |
| Feed | Subtracted | Food products fed to animals, including by-products of processing such as oilseed cake and bran. |
| Animal products | Added | Meat, offal, animal fats, milk, eggs, honey, fish, and seafood produced in the country. |
| Tourist consumption | Subtracted | Food eaten by visitors from other countries. |
| Data adjustments | Subtracted | Differences needed to make the stages add up to the food available to eat. |
| Food | Ends the chain | Food available for people to eat, before waste in shops and households. |

Food enters the chain through crop production, imports, and animal products.
Stock variation can go either way: food put into stocks leaves the chain, and food taken out of stocks enters it.
In principle, all other stages only take food out of the chain.
However, some of them can be negative, and then they add to the chain instead:

- Processing, net, in kilograms only, when the processed products weigh more than what went into processing, for example because breweries add water.
In calories and protein, processing should not add to the chain, because no factory can create calories or protein.
Where our figures show more calories or protein coming out of processing than going in, we set processing to zero and move the difference to data adjustments (see [the section on processing](#Processing-never-adds-calories-or-protein)).
- Data adjustments, when the other stages add up to less than the food available to eat.
In calories and protein, data adjustments include the difference moved from processing, which adds to the chain.

We build the chain three times, once in each unit:

- Energy, in kilocalories per person per day.
- Protein, in grams of protein per person per day.
- Mass, in kilograms per person per day.
This is the balance in tonnes, with no conversion at all.

The chart shows the chain of the world in 2023, in kilocalories.

In [2]:
def plot_chain(tb, country, year, unit, title):
    """Plot the chain of a country in a year: each stage adds to the chain or subtracts from it, and it ends on food."""
    row = tb[(tb["country"] == country) & (tb["year"] == year)].iloc[0]
    changes = [float(sign * row[stage]) for stage, (_, sign) in STAGES.items()]
    assert abs(sum(changes) - row["food"]) < 0.001 * row["food"], "The chain does not end on food."
    fig = go.Figure(
        go.Waterfall(
            x=[name for name, _ in STAGES.values()] + ["Food"],
            y=changes + [0],
            measure=["relative"] * len(changes) + ["total"],
            text=["0" if change == 0 else f"{change:+,.0f}" for change in changes] + [f"{row['food']:,.0f}"],
            textposition="outside",
            hovertemplate="%{x}: %{text} " + unit + "<extra></extra>",
            increasing=dict(marker=dict(color="#2ca02c")),
            decreasing=dict(marker=dict(color="#d62728")),
            totals=dict(marker=dict(color="#1f77b4")),
        )
    )
    fig.update_yaxes(title_text=f"{unit} per person per day")
    fig.update_layout(title=title, showlegend=False, height=520, margin=dict(t=80))
    fig.show()


# The figures for the world in 2023 given in the introduction, in kcal per person per day.
world_2023 = energy[(energy["country"] == "World") & (energy["year"] == 2023)].iloc[0]
error = "The world's crop production or food in 2023 have changed."
assert (round(world_2023["crop_production"], -3), round(world_2023["food"], -3)) == (6000, 3000), error

plot_chain(energy, country="World", year=2023, unit="kcal", title="The world's food supply chain in 2023, in kilocalories")

## FAOSTAT's food balances

FAOSTAT publishes two datasets with food balances:

- The [Food Balance Sheets](https://www.fao.org/faostat/en/#data/FBS) (FAOSTAT code FBS), which start in 1961.
- The [Supply Utilization Accounts](https://www.fao.org/faostat/en/#data/SCL) (FAOSTAT code SCL), which start in 2010.

From now on, we refer to each dataset by its code.

Both datasets are indexed by country, year, item, and element.

!!! note ""
    In FAOSTAT's terms, an **item** is, roughly, the food product, and an **element** is, roughly, the metric.
    For example, the tonnes of wheat fed to animals correspond to item "Wheat" and element "Feed".

We use SCL, because it has two advantages:

- SCL reports individual products, while FBS reports groups.
SCL has "Wheat", "Bread", and other items processed from wheat, as separate items; FBS merges them into one item, "Wheat and products".
- SCL includes by-products that FBS has no items for: oilseed cakes, brans, and gluten feed.
When oilseeds such as soybeans are crushed for their oil, the solids left over are called cake.
Cake is rich in protein, and it is one of the main things farm animals eat.
With cakes as items, the path from oilseeds to cake to animals shows up under feed, where it belongs.
In FBS, the crushed oilseeds count as processing, and the cake is never counted at all.

The disadvantage of using SCL instead of FBS is coverage:

- SCL starts in 2010.
- SCL has no fish or seafood.

Both datasets also miss some countries, such as Japan, Sudan, and Somalia, which FAO left out of its latest release.

The following sections explain how we build the chain from SCL.
We also build a similar chain from FBS, which we use only for a chart of the world in 1968; [the last section](#The-chain-built-from-the-Food-Balance-Sheets) explains how it differs.

In [3]:
# The coverage of the chain built from SCL.
assert (energy["year"].min(), energy["year"].max()) == (2010, 2023), "The years of the chain built from SCL have changed."
error = "Japan, Sudan or Somalia are now in the chain built from SCL."
assert not energy["country"].isin(["Japan", "Sudan", "Somalia"]).any(), error

## The balance of each item

FAO's starting point for SCL is that, for every item, country, and year, the total supply of the item must equal its total use.
FAO expresses this as an identity that sets total supply equal to total utilization ([FAO 2025, Food balance sheets and supply utilization accounts resource handbook](https://openknowledge.fao.org/items/30e641b1-6d57-4998-8e8a-395727aa4307), section 1.1).
In tonnes:

`production + imports - exports - stock variation = food + feed + seed + processing + other uses + losses + tourist consumption + residuals`

FAO balances each item by spreading any imbalance across its elements, and where that is not possible, it puts what is left in the residuals.
So the identity holds by construction.

When SCL does not report an element for an item, we treat it as zero.
FAO builds each balance as a whole, so an element missing from a balance is almost always one that does not apply to the item.
If a real value were missing, the item's balance would not close, and our code stops when more than 1% of the item balances do not close.

We first correct two errors in SCL:

- Since 2021, FAO imputes Denmark's molasses production as equal to all the sugar beet that Denmark processes, and balances it with a feed use of about the same size.
We set the production to the last reported figure (of 2020), and the feed to the value that closes the molasses balance.
- For 2010, FAO reports negative wheat exports for Iraq, a data artifact.
We set them to zero, and move the difference to the residuals.

In [4]:
# FAOSTAT's SCL has about 12 million rows, so we read its published file directly, and only the columns we need.
scl = pd.read_feather(
    f"{CATALOG_URL}/{FAOSTAT_SCL_DATASET}/faostat_scl.feather",
    columns=["country", "year", "fao_item", "element_code", "value"],
)
# Elements used in this notebook: food in million kilocalories and in tonnes, production, processing, feed, and exports.
ELEMENTS = {
    "000261": "food_million_kcal",
    "005141": "food_tonnes",
    "005510": "production",
    "005023": "processing",
    "005520": "feed",
    "005910": "exports",
}
# Keep only those elements, and the countries of the chain.
countries = set(energy["country"]) - set(REGIONS)
scl = scl[scl["element_code"].isin(list(ELEMENTS)) & scl["country"].isin(countries)]
scl = scl.astype({"country": str, "fao_item": str, "element_code": str, "value": float})
# One row per country, year, and item, with one column per element.
scl = scl.pivot_table(index=["country", "year", "fao_item"], columns="element_code", values="value").rename(columns=ELEMENTS)

# The two errors that we correct: from 2021, Denmark's molasses production equals all the sugar beet that Denmark
# processes, and its molasses feed is about as large; and Iraq's wheat exports in 2010 are negative.
for year in [2021, 2022, 2023]:
    molasses = scl.loc[("Denmark", year, "Molasses"), ["production", "feed"]]
    beet_processed = float(scl.loc[("Denmark", year, "Sugar beet"), "processing"])
    assert molasses["production"] == beet_processed, "Denmark's molasses production has changed."
    assert abs(molasses["feed"] / beet_processed - 1) < 0.05, "Denmark's molasses feed has changed."
assert float(scl.loc[("Iraq", 2010, "Wheat"), "exports"]) < 0, "Iraq's wheat exports in 2010 are no longer negative."

### Items that are never food

Fuel ethanol, and a few other items that are never food (such as castor oil and jojoba oil), are left out of the chain.

The crops sent to distilleries already leave the chain where FAO records them: as other uses of the crop (uses other than food or feed, such as biofuels) in Brazil and the United States, or as processing of molasses in Pakistan and India.
Counting the ethanol itself as a processed product would cause three errors:

- Its calories would be credited back into processing, making processing appear to create calories.
- The same calories would be counted again, as ethanol exports and ethanol burned.
- The fuel that importing countries buy would enter their chain as imports.

### Fish and seafood

SCL has no fish or seafood, so we add the fish items of FBS for the countries and years that SCL covers.
They go through the same rules as any other item, except for their densities (see [the section on fish densities](#Densities-of-fish)).

FBS repeats its 2021 fish tonnes in 2022 and 2023 for every country.
So recent fish figures are older than the rest of the chain.

## Converting tonnes into calories and protein

### Densities from FAO's food figures

SCL reports every element in tonnes.
Food is the exception: it is also reported in calories (the element "Calories/Year") and in protein ("Proteins/Year").

The density of an item is its food nutrient divided by its food quantity.
By food nutrient we mean either calories or protein.
The resulting density is measured in kilocalories per 100 g, or in grams of protein per 100 g.

FAO uses one density per item in every country and year: for example, 334 kcal per 100 g for wheat grain, 348 for maize, and 64 for potatoes.
There are only nine exceptions: nine country-items with a density of their own, such as potatoes in Peru (91 kcal per 100 g, instead of the usual 64) and milled rice in Bangladesh (360, instead of 349).

So each item gets one density: its food nutrient divided by its food tonnes, both summed over all countries and all years.
The nine exceptions are not used to calculate that density.
Each of them is converted with its own density instead, from its own food figures.

FAO takes these densities from its [nutrient conversion table](https://doi.org/10.4060/cc9678en): for the items in that table, FAO's food calories are the table's value times the item's edible-portion coefficient, the share of the item that people eat.

We use the density of an item to convert all its other elements: production, imports, exports, stock variation, seed, losses, other uses, processing, feed, tourist consumption, and residuals.
Food keeps the calories and protein that FAO reports for it.
Where FAO reports no food nutrient for an item, food is converted like all other elements, as quantity times density.

The mass table does not need any densities: SCL already reports everything as mass, and we only convert tonnes to kilograms.

In [5]:
# Density of each item in each country and year, in kcal per 100 g (a tonne is 10,000 times 100 g).
food = scl.dropna(subset=["food_million_kcal", "food_tonnes"]).reset_index()
# Below 1,000 tonnes of food, FAO's rounding distorts the density.
food = food[food["food_tonnes"] >= 1000]
food["density"] = 1e6 * food["food_million_kcal"] / (1e4 * food["food_tonnes"])

# Country-items whose density differs from the usual density of the item (its median over all countries and years)
# by more than 1% (and by more than 0.5 kcal per 100 g).
usual_density = food.groupby("fao_item")["density"].transform("median")
differs = (food["density"] - usual_density).abs() > (0.01 * usual_density).clip(lower=0.5)
exceptions = set(zip(food[differs]["country"], food[differs]["fao_item"]))
error = "The country-items with a density of their own have changed."
assert exceptions == {
    ("Afghanistan", "Wheat and meslin flour"),
    ("Bangladesh", "Rice, milled"),
    ("Ethiopia", "Other vegetables provisionally preserved"),
    ("Namibia", "Skim milk of cows"),
    ("North Korea", "Rice, milled"),
    ("Peru", "Cassava, fresh"),
    ("Peru", "Meat of chickens, fresh or chilled"),
    ("Peru", "Potatoes"),
    ("Peru", "Rice, milled"),
}, error


def item_density(item, country=None):
    """Density of an item, in kcal per 100 g: its food calories divided by its food quantity, both summed over all
    countries and years (without the nine exceptions), or over the years of one country."""
    rows = food[(food["fao_item"] == item) & ((food["country"] == country) if country else ~differs)]
    return 1e6 * rows["food_million_kcal"].sum() / (1e4 * rows["food_tonnes"].sum())


# The densities given in the text.
densities = [round(item_density(item)) for item in ["Wheat", "Maize (corn)", "Potatoes", "Rice, milled"]]
assert densities == [334, 348, 64, 349], "The densities of wheat, maize, potatoes or milled rice have changed."
assert round(item_density("Potatoes", country="Peru")) == 91, "The density of potatoes in Peru has changed."
assert round(item_density("Rice, milled", country="Bangladesh")) == 360, "The density of milled rice in Bangladesh has changed."

### Items that people hardly eat

Some items have too little food use to give a reliable density.
They are mainly cakes, brans, gluten feed, and oil refining residues, which are hardly eaten, but also centrifugal cane sugar, hemp oil, hempseed, and tallow.
We give them fixed densities instead, from two sources:

- FAO's nutrient conversion table, where it has the item: cereal brans, centrifugal cane sugar, hemp oil, and tallow.
Acid oils from refining and residues of fatty substances, which the table does not have, take its value for palm oil.
- [Feedipedia](https://www.feedipedia.org), for the oilseed cakes, gluten feed, bran of pulses, and hempseed, which FAO's table does not cover.

Feedipedia is an online encyclopedia of animal feeds, run by the French National Research Institute for Agriculture, Food and Environment (INRAE), the French Agricultural Research Center for International Development (CIRAD), the French Association for Animal Production (AFZ), and FAO.
For each feed, it publishes laboratory analyses averaged over many samples: how much of the feed is water, and how much of the rest (the dry matter) is protein, fat, fibre, and ash.

Feedipedia also gives energy values, but not the kind we need.
It gives gross energy: the heat released when a dried sample of the feed is burned completely in a laboratory.
That is all the energy stored in the feed, including energy that a person could not get from it.
FAO's food values count only the energy that people can get from food.
So we calculate the energy from the composition, with the formula FAO uses for its own table.
That way, a cake's calories are counted in the same way as those of the crop and the oil it comes from.
For each item:

- Each component, given in % of dry matter, is converted to grams per 100 g of the feed as it is.
- Available carbohydrate is what is left after water, protein, fat, ash, and fibre (zero if the published values add up to more than 100 g).
- Energy is 4 kcal per gram of protein, 9 per gram of fat, 4 per gram of available carbohydrate, and 2 per gram of fibre.
The protein density is the grams of protein.

Feedipedia gives more than one measure of fibre.
We use neutral detergent fibre, or crude fibre where Feedipedia gives no neutral detergent fibre.

For example, Feedipedia's table ["Soybean meal, low protein, type 44-46 and similar"](https://www.feedipedia.org/node/11682) gives a dry matter of 87.7%, and, as shares of the dry matter, 49.5% crude protein, 1.9% ether extract (fat), 7.4% ash, and 14.8% neutral detergent fibre.
Per 100 g of soybean cake, that is 43.4 g of protein, 1.7 g of fat, 6.5 g of ash, and 13.0 g of fibre, which leaves 23.2 g of available carbohydrate: 307 kcal.

Where Feedipedia has several types of the same cake, we use the type that matches what SCL records.
Cakes with no Feedipedia table of their own take the average of the cakes that have one.


In [6]:
# Composition of soybean cake, from Feedipedia's table "Soybean meal, low protein, type 44-46 and similar": dry matter,
# in % of the feed as it is, and protein, fat, ash, and fibre (neutral detergent fibre), in % of dry matter.
DRY_MATTER = 87.7
COMPOSITION = {"protein": 49.5, "fat": 1.9, "ash": 7.4, "fibre": 14.8}

# Grams of each component per 100 g of the feed as it is.
grams = {component: share * DRY_MATTER / 100 for component, share in COMPOSITION.items()}
# Available carbohydrate is what is left after water, protein, fat, ash, and fibre.
grams["carbohydrate"] = max(0, DRY_MATTER - sum(grams.values()))
kcal = 4 * grams["protein"] + 9 * grams["fat"] + 4 * grams["carbohydrate"] + 2 * grams["fibre"]

# The figures given in the text.
error = "The composition or the energy of soybean cake have changed."
expected = {"protein": 43.4, "fat": 1.7, "ash": 6.5, "fibre": 13.0, "carbohydrate": 23.2}
assert {component: round(value, 1) for component, value in grams.items()} == expected, error
assert round(kcal) == 307, error

### Crops that are mostly processed

FAO's table values each crop at the part that people eat, through its edible-portion coefficient: for example, sugar cane as its juice, paddy as husked rice, and rapeseed as its oil.
FAO uses this table to count the calories in people's food, so the parts that people do not eat count for nothing.

That value is right for the little of these crops that people eat as harvested.
But almost all of them go to factories, which get more out of them than the part that people eat.
Rapeseed is the clearest case.
FAO's value counts only its oil, but crushing rapeseed gives oil and also cake, which is rich in protein and is fed to animals.
So at FAO's values, the calories in the cake would seem to come from nowhere.
The same happens with other crops that are crushed or milled, such as sunflower seed in Ukraine and coconuts in the Pacific islands: processing would appear to create calories.

So we value these crops by their products: sugar cane, sugar beet, paddy rice, palm kernels, oil palm fruit, rapeseed, cotton seed, sunflower seed, coconuts, olives, and maize germ.
Each year, the density of a crop is the nutrient in everything the world's factories make from it, divided by the tonnes of the crop that the world processes.
That world density is applied to every country.

We call a crop and the products made from it a family.
These rules decide which products count:

- Each product counts once.
Some products are processed further within the family, such as copra, which is pressed into coconut oil and cake.
We count all the products, and subtract the copra that was pressed, because its calories are already in the oil and the cake.
- A family ends at the products made from its crop alone: for example, sugar and molasses, milled rice and rice bran products, and the oils and cakes of oilseeds.
Later products, such as sweets or rice flour, are left out, because the calories they get from the crop are already counted in the sugar or rice they are made from.
- Sugar cane and sugar beet get one density together, because SCL records some of their products, such as raw sugar, refined sugar, and molasses, without saying which crop they come from.
- Palm kernels get their density before oil palm fruit.
Palm kernels come from oil palm fruit and are then crushed for their oil and cake, so the oil palm fruit family counts its kernels at that density.

Which products are made from each crop follows FAO's commodity trees ([FAO, Technical Conversion Factors for Agricultural Commodities](https://www.fao.org/fileadmin/templates/ess/documents/methodology/tcf.pdf)).

### Densities of fish

The fish items that we take from FBS differ from the other items in two ways:

- FBS reports fish as groups, such as pelagic fish or demersal fish.
The species in a group change from year to year, and so does the density.
So fish keep the density of each year.
- FBS rounds its tonnes to the nearest 1,000 tonnes, so small quantities are imprecise, and can even be zero.
FAO's food per person, in kilograms and in calories, is not calculated from those rounded tonnes: where FBS rounds a fish item's food to zero tonnes, its food per person is often above zero.
So, for fish, the density is FAO's food calories per person divided by its food kilograms per person, instead of food calories divided by food tonnes (and the same for protein).

## From items to the stages of the chain

Adding up the production of all items would count the same calories many times, because sugar is made from sugar cane, and bread from wheat.
So each item has a role, which decides where its production goes.
FAO's own item groups put every SCL item in one of four groups ("Crops, primary", "Livestock primary", "Crops processed", and "Livestock processed"), which map to three roles:

- Crop: the production of crops is the stage "crop production", the start of the chain.
- Animal: the stage "feed" subtracts the crops fed to animals, and the stage "animal products" adds back the meat, milk, and eggs that the animals produce.
The production of animal items is that second stage.
- Processed: factories turn crops into sugar, oils, flour, and other products.
The stage "processing" is what goes into factories (the element "processing" of all items) minus what comes out (the production of processed items).
So it counts only the calories that enter factories and do not come back as food.

The role only decides where the production of an item goes.
Its uses (seed, losses, other uses, feed, and processing) go to their own stages, whatever the role.

There are two special cases:

- Cotton seed counts as a crop, although FAO groups it with processed crops.
Cotton is grown for its fibre, and the seeds are separated from the fibre after the harvest.
SCL does not record the cotton harvest itself, only the seeds and what is made from them.
If cotton seed counted as a processed product, its calories would come out of processing with nothing going in.
As a crop, its calories enter the chain in crop production.
- Fish oil counts as an animal product, like the fish it comes from.
FBS records no fish going into processing, so counting fish oil as a processed product would make processing create calories.

### Data adjustments

Data adjustments are the difference between what the other stages add up to and the food available to eat, so that the chain ends exactly on food.
That difference comes from:

- FAO's own residuals.
- Rounding.
Food keeps the calories and protein that FAO reports, while the other stages use FAO's rounded tonnes times the density of the item.
This matters most for fish.
- A difference of valuation, for the crops valued by their products that people both eat as harvested and process: mostly coconuts and sugar cane.
The tonnes that people eat leave the earlier stages at the density that the crop gets from its products, but arrive in food at FAO's density for the part people eat.
- In calories and protein, any gain in processing (see [the next section](#Processing-never-adds-calories-or-protein)).
- For the world, the gap between imports and exports, and tourist consumption (see [the section on the world](#World-trade-and-tourist-consumption)).

For the world, data adjustments are 43 kcal per person per day in 2023, which is 1.4% of food.
In many countries, data adjustments are a larger share of food.
In 2023, 28 of the 176 countries with data had data adjustments, positive or negative, larger than 10% of their food in calories.
The table lists these countries, and also those where data adjustments are larger than 10% of food in protein.
A positive value is subtracted from the chain, and a negative value is added to it.

In [7]:
# The figures given above, for 2023.
countries_2023 = energy[(energy["year"] == 2023) & ~energy["country"].isin(REGIONS)]
protein_2023 = protein[(protein["year"] == 2023) & ~protein["country"].isin(REGIONS)]
assert len(countries_2023) == 176, "The number of countries with data in 2023 has changed."

# Data adjustments of the world, in kcal per person per day and as a share of food.
adjustments = (round(world_2023["data_adjustments"]), round(100 * world_2023["data_adjustments"] / world_2023["food"], 1))
assert adjustments == (43, 1.4), "The world's data adjustments in 2023 have changed."
# Countries where data adjustments are more than 10% of food.
large_adjustments = countries_2023["data_adjustments"].abs() > 0.1 * countries_2023["food"]
assert large_adjustments.sum() == 28, "The number of countries with large data adjustments has changed."

# Data adjustments as a share of food, in calories and in protein, in the countries where they are more than 10% of
# food in either, sorted by their share in calories.
shares = countries_2023[["country", "data_adjustments", "food"]].merge(
    protein_2023[["country", "data_adjustments", "food"]], on="country", suffixes=("_energy", "_protein")
)
shares["Calories"] = 100 * shares["data_adjustments_energy"] / shares["food_energy"]
shares["Protein"] = 100 * shares["data_adjustments_protein"] / shares["food_protein"]
shares = shares[(shares[["Calories", "Protein"]].abs() > 10).any(axis=1)].sort_values("Calories", key=abs, ascending=False)
table = shares[["country", "Calories", "Protein"]].rename(
    columns={"country": "Country", "Calories": "In calories", "Protein": "In protein"}
)
for column in ["In calories", "In protein"]:
    table[column] = [f"{share:+.0f}%" for share in table[column]]
html = table.to_html(index=False)
summary = "Countries where data adjustments are more than 10% of food in 2023, as a share of food"
display(HTML(f"<details><summary>{summary}</summary>{html}</details>"))


Country,In calories,In protein
Djibouti,+310%,+164%
Guyana,-63%,-73%
Vanuatu,+46%,+22%
Ukraine,-44%,-40%
Uruguay,-39%,-62%
Belgium,-25%,-34%
Romania,-24%,-26%
Netherlands,-23%,-17%
Belize,-22%,-28%
Paraguay,-22%,-32%


### Processing never adds calories or protein

A mill, a crusher, or a brewery cannot put more calories or protein into its products than came in.
Where the converted products exceed the converted inputs, the excess is a failure of the data or of the densities (the causes are in [the section on known problems](#Known-problems)), not a feature of the country's food chain.

So we move the gain to data adjustments, and processing shows zero.
Only gains move.
The rule applies to each country and region in each year.
In 2023, it applies to 46 of the 176 countries in calories, and to 31 in protein.
The world shows a loss in both, so it is unchanged.

The mass table is left alone.
In tonnes, a gain in processing can be real, because breweries and bottlers add water: in Namibia in 2023, 22 thousand tonnes of malt became 174 thousand tonnes of beer.
So processing there is the net change in weight, in either direction.

### World trade and tourist consumption

The world as a whole trades with no one, so world imports and exports should be equal.
In the data they differ, because each is the sum of what countries report.
We set world exports equal to world imports, which FAO considers the better-documented side ([FAO 2025, Food balance sheets and supply utilization accounts resource handbook](https://openknowledge.fao.org/items/30e641b1-6d57-4998-8e8a-395727aa4307), section 6.1), and the difference goes to data adjustments.
Other regions do trade with the rest of the world, so we leave them as they are.

Tourist consumption is food eaten in a country by people who live in another country.
Every visitor lives in some country, so the world as a whole has no visitors, and world tourist consumption should be zero.
In the data it is not, so we set it to zero, and the difference goes to data adjustments.
Countries and other regions keep their tourist consumption, because for them it is real.

### Regions

We drop FAO's own regional aggregates, and build [our regions](https://ourworldindata.org/world-region-map-definitions) (the world, continents, income groups, and the European Union) from countries.
Each stage of a region is the sum of that stage over the member countries that have an SCL balance that year, and the region's population is the sum of the same countries' populations.
The rule on processing is then applied to the region's own stages.
So the processing and data adjustments of a region are not always the sums of those shown for its countries: within the region, the gains of some countries offset the losses of others before the rule applies.

A region-year is dropped when the countries with a balance hold less than 80% of the region's population, so that a value labeled "Africa" is never built from a small part of Africa.
"Low-income countries" is left out altogether: SCL covers just under or just over 80% of its population, depending on the year (FAO's latest release leaves out Sudan, Somalia, and some other countries), so its series would have holes.

### Values per person per day

All stages are divided by population and by 365 days.
The population estimates come from a long-run dataset [maintained by Our World in Data](https://ourworldindata.org/population-sources).

In [8]:
# The figures given above, for 2023.
# Countries where a gain in processing is moved to data adjustments, so that processing shows zero.
error = "The number of countries with a gain in processing has changed."
assert ((countries_2023["processing_net"] == 0).sum(), (protein_2023["processing_net"] == 0).sum()) == (46, 31), error
world_protein_2023 = protein[(protein["country"] == "World") & (protein["year"] == 2023)].iloc[0]
error = "The world no longer shows a loss in processing."
assert (world_2023["processing_net"] > 0) and (world_protein_2023["processing_net"] > 0), error

# Malt processed and beer produced in Namibia in 2023, in thousand tonnes.
malt = float(scl.loc[("Namibia", 2023, "Malt, whether or not roasted"), "processing"])
beer = float(scl.loc[("Namibia", 2023, "Beer of barley, malted"), "production"])
assert (round(malt / 1e3), round(beer / 1e3)) == (22, 174), "Namibia's malt or beer have changed."

# The regions of the chain.
error = "The regions of the chain have changed."
assert set(energy["country"]) & set(REGIONS) == set(REGIONS) - {"Low-income countries"}, error

## Known problems

### Processing would create calories in some countries

We move the gains in processing to data adjustments, but their causes remain.
The gains do not leave the chart: they appear in data adjustments as calories entering the chain (more than half of food in Guyana in 2023).
The same causes also inflate or shrink the real losses of other countries, which stay in processing.
So processing is the least certain stage.

Some gain is moved in 20% of all country-years in calories, and in 14% in protein.
For the world, processing loses 105 kcal per person per day in 2023, which is 1.8% of crop production.

The gains have four causes:

- FAO's own quantities are inconsistent.
Guyana's rice bran is 30% of the paddy it mills.
In 2022, Bolivia's soybean cake weighs more than the beans crushed.
We correct only one such error, Denmark's molasses (see [the section on the balance](#The-balance-of-each-item)).
- Factories that make more from a crop than the world average, which is the density that all countries get for the [crops that are mostly processed](#Crops-that-are-mostly-processed): oil palm in Honduras and Papua New Guinea, sunflower seed in Ukraine, rice in Uruguay, and sugar beet in Ukraine, the Netherlands, and Germany (beet holds more sugar than cane, and the two share one density).
A country whose factories make less than the world average shows the difference as a processing loss instead: coconuts in Tonga and Micronesia, and oil palm in Malaysia.
Both mix real differences between factories with differences in what FAO records.
- FAO's density for a raw item is below what its products contain: raw milk against the powders, cheese, butter, and casein made from it (New Zealand in protein; Uruguay and the Netherlands in calories).
- Products with no recorded input: oilseed flours (Bolivia, in protein).

### Feed only counts feed that passes through the balance

Grass, pasture, and forage are not SCL items, so everything that grazing animals eat from pasture is missing from the chain.
Wild-caught fish count as animal products with no feed at all.

So in fishing and grazing countries (Iceland, Mongolia, and several island states), animal products can exceed feed.
This happens in 14% of country-years in calories, and in 24% in protein.
The world still has more recorded feed than animal products: 2,048 against 635 kcal per person per day in 2023, and 103 against 43 g of protein.

The balance alone cannot tell how much of a country's gap these missing inputs explain.

### Fish oil is partly counted twice

We count fish at the density of the fish that people eat, and fish oil on top of it.
Where the fish that people eat are lean, that density is about the energy of the fishmeal, and adding the oil is right.
Where people eat fatty fish, that density already exceeds the energy of the fishmeal, so part of the oil pressed from the fish is counted a second time.
This is minor for the world, and larger in a few oil producers whose food fish are fatty, such as Denmark and Norway.
Protein is unaffected.

In tonnes, the oil is inside the fish tonnes in every producer, so the mass table counts it twice.

We weighed two alternatives and rejected both:

- Leaving the oil out would miss the energy it adds where the food fish are lean, and the oil that importers buy.
- Counting it in calories but not in tonnes would remove the double counting from the mass table only, would miss the oil that importers buy in tonnes, and would make one table an exception.

### FAO records the crops sent to distilleries inconsistently

Where FAO records the crops sent to distilleries as processing, the calories turned into fuel appear as a processing loss rather than under other uses.
This happens with molasses in Eswatini, Mauritius, South Africa, Pakistan, and India, and with cereals in Slovakia and Moldova.

In [9]:
# The figures given above.
all_countries = energy[~energy["country"].isin(REGIONS)]
all_countries_protein = protein[~protein["country"].isin(REGIONS)]

# Data adjustments of Guyana in 2023 (negative values enter the chain), as a share of food.
guyana = countries_2023[countries_2023["country"] == "Guyana"].iloc[0]
assert -guyana["data_adjustments"] > 0.5 * guyana["food"], "Guyana's data adjustments are no longer more than half of food."
# Share of country-years where a gain in processing is moved to data adjustments.
gains = (round(100 * (all_countries["processing_net"] == 0).mean()), round(100 * (all_countries_protein["processing_net"] == 0).mean()))
assert gains == (20, 14), "The share of country-years with a gain in processing has changed."
# Processing of the world in 2023, in kcal per person per day and as a share of crop production.
processing = (round(world_2023["processing_net"]), round(100 * world_2023["processing_net"] / world_2023["crop_production"], 1))
assert processing == (105, 1.8), "The world's processing in 2023 has changed."

# Rice bran produced in Guyana in 2023, as a share of the paddy it mills.
bran = float(scl.loc[("Guyana", 2023, "Bran of rice"), "production"])
paddy = float(scl.loc[("Guyana", 2023, "Rice"), "processing"])
assert round(100 * bran / paddy) == 30, "Guyana's rice bran is no longer 30% of the paddy it mills."
# Soybean cake produced in Bolivia in 2022, and soybeans crushed.
cake = float(scl.loc[("Bolivia", 2022, "Cake of soya beans"), "production"])
beans = float(scl.loc[("Bolivia", 2022, "Soya beans"), "processing"])
assert cake > beans, "Bolivia's soybean cake no longer weighs more than the beans crushed."
# Share of country-years where animal products exceed feed (by more than 1 kcal, or 0.01 g of protein).
exceeds = (
    round(100 * (all_countries["animal_products"] - all_countries["feed"] > 1).mean()),
    round(100 * (all_countries_protein["animal_products"] - all_countries_protein["feed"] > 0.01).mean()),
)
assert exceeds == (14, 24), "The share of country-years where animal products exceed feed has changed."
for country in ["Iceland", "Mongolia"]:
    row = countries_2023[countries_2023["country"] == country].iloc[0]
    assert row["animal_products"] > row["feed"], f"Animal products no longer exceed feed in {country}."
# Feed and animal products of the world in 2023, in kcal and in grams of protein per person per day.
world_feed = [round(world[stage]) for world in [world_2023, world_protein_2023] for stage in ["feed", "animal_products"]]
assert world_feed == [2048, 635, 103, 43], "The world's feed or animal products in 2023 have changed."

## The chain built from the Food Balance Sheets

We also build the chain from FBS.
FBS starts in 1961, long before SCL.
We use this chain for one static chart only: the world in 1968.

Here, FBS is the combination of two FAOSTAT datasets: [Food Balances (-2013, old methodology and population)](https://www.fao.org/faostat/en/#data/FBSH), which we use from 1961 to 2009, and [Food Balances (2010-)](https://www.fao.org/faostat/en/#data/FBS), which we use from 2010.

The chain is built as for SCL, with these differences:

- Items.
FBS reports groups, such as "Wheat and products", which holds wheat grain, flour, bread, and pasta, with their quantities expressed as the weight of the wheat used to produce them.
We assign the role of each item by hand.
Fish and seafood are FBS items, so nothing is added from another dataset.
We leave out the items "Alcohol, Non-Food" and "Meat, Aquatic Mammals", for which FAO reports no food calories.
- Oil palm.
In FBS, the whole harvest of the oil palm is recorded under the item "Palm kernels", which has almost no food use, so no density can be calculated for it.
We leave that item out, and the oil palm harvest enters the chain through the items "Palm Oil" and "Palmkernel Oil", which count as crops.
- Stock variation.
FBS reports stock variation only from 2010.
So we calculate it for all years from the balance, as production plus imports, minus exports, minus domestic supply.
- Densities.
Each item gets a density for each country and year: its food nutrient divided by its food quantity.
A density is rejected when it is physically impossible: more energy than pure fat (920 kcal per 100 g), or more than 100 g of protein per 100 g.
A rejected or missing density is replaced by the country's median density for the item over all years, or else by the item's median over all countries and years.
A density of exactly zero is replaced in the same way, because FAO may have rounded a small food nutrient down to zero.
Food keeps the calories and protein that FAO reports where the density is not replaced.
- Crops that are mostly processed.
All densities come from food.
So crops that are mostly processed, such as sugar cane, are valued only by the part that people eat, and processing can appear to create calories.
We do not move those gains to data adjustments: processing is negative in more than a third of country-years (in Brazil, about -670 kcal per person per day in 2023).
- Oilseed cakes.
Oilseed cakes are not FBS items.
So their calories and protein appear as lost in processing, instead of as feed.
For the world in 2023, FBS has 39 g of protein per person per day in feed and 56 g in processing, while SCL has 103 g in feed and 4 g in processing.
- Regions.
The tonnes and the food nutrients of a region are the sums over its member countries, and the region then gets densities of its own.
The rule on population removes "Low-income countries" before 2010 and in 2023, and Oceania from 2002 to 2009.

The chart compares the two chains for the world in 2023.
Bars to the right add to the chain, bars to the left subtract from it, and the last bars are the food at the end of the chain.

In [10]:
# Load the chain built from FBS, in kilocalories and in grams of protein per person per day.
energy_fbs = load_table(CHAIN_FBS_DATASET, "energy")
protein_fbs = load_table(CHAIN_FBS_DATASET, "protein")
assert (energy_fbs["year"].min(), energy_fbs["year"].max()) == (1961, 2023), "The years of the chain built from FBS have changed."

# Share of country-years where processing is negative, and processing in Brazil in 2023, in kcal per person per day.
countries_fbs = energy_fbs[~energy_fbs["country"].isin(REGIONS)]
assert (countries_fbs["processing_net"] < 0).mean() > 1 / 3, "Processing is no longer negative in more than a third of country-years."
brazil = countries_fbs[(countries_fbs["country"] == "Brazil") & (countries_fbs["year"] == 2023)].iloc[0]
assert round(brazil["processing_net"], -1) == -670, "Brazil's processing in 2023 has changed."

# Protein in feed and in processing for the world in 2023, in grams per person per day, from FBS and from SCL.
world_protein_fbs = protein_fbs[(protein_fbs["country"] == "World") & (protein_fbs["year"] == 2023)].iloc[0]
feed_and_processing = [round(world[stage]) for world in [world_protein_fbs, world_protein_2023] for stage in ["feed", "processing_net"]]
assert feed_and_processing == [39, 56, 103, 4], "The world's protein in feed or in processing in 2023 has changed."

# Years without data for the regions removed by the rule on population.
years = set(range(1961, 2024))
missing_years = {region: years - set(energy_fbs[energy_fbs["country"] == region]["year"]) for region in ["Low-income countries", "Oceania"]}
error = "The years removed by the rule on population have changed."
assert missing_years == {"Low-income countries": set(range(1961, 2010)) | {2023}, "Oceania": set(range(2002, 2010))}, error

# The world in 2023, in the chain built from FBS.
world_energy_fbs = energy_fbs[(energy_fbs["country"] == "World") & (energy_fbs["year"] == 2023)].iloc[0]

# The figures given below, in kcal and in grams of protein per person per day.
error = "The world's food in 2023 is no longer about 3,000 kcal and 93 g of protein in both chains."
assert [round(world["food"], -2) for world in [world_2023, world_energy_fbs]] == [3000, 3000], error
assert [round(world["food"]) for world in [world_protein_2023, world_protein_fbs]] == [93, 93], error
error = "The world's crop production, feed, or processing in 2023 have changed."
figures = [round(world[stage]) for stage in ["crop_production", "feed", "processing_net"] for world in [world_2023, world_energy_fbs]]
assert figures == [5998, 5219, 2048, 1386, 105, 341], error

# Each stage with its sign in the chain (positive if it adds to the chain, negative if it subtracts from it), and the
# food at the end, for the world in 2023, in both chains.
names = [name for name, _ in STAGES.values()] + ["Food"]
units = ["Kilocalories per person per day", "Grams of protein per person per day"]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04, subplot_titles=units)
for col, worlds in enumerate([(world_2023, world_energy_fbs), (world_protein_2023, world_protein_fbs)], start=1):
    for label, world, color in zip(["Built from SCL", "Built from FBS"], worlds, ["#2a78d6", "#eb6834"]):
        values = [float(sign * world[stage]) for stage, (_, sign) in STAGES.items()] + [float(world["food"])]
        bars = go.Bar(
            y=names,
            x=values,
            orientation="h",
            name=label,
            legendgroup=label,
            showlegend=col == 1,
            marker_color=color,
            hovertemplate="%{y}: %{x:,.0f}<extra>" + label + "</extra>",
        )
        fig.add_trace(bars, row=1, col=col)
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(zeroline=True, zerolinecolor="#888888", tickformat=",")
fig.update_layout(
    title="The world's food supply chain in 2023, built from SCL and from FBS",
    barmode="group",
    bargroupgap=0.1,
    height=620,
    legend=dict(orientation="h", y=-0.08),
    margin=dict(t=100),
)
fig.show()


Both chains end at almost the same food: about 3,000 kcal and 93 g of protein per person per day.
On the way, they differ for the reasons listed above.
The chain built from FBS has less crop production (5,219 against 5,998 kcal per person per day), less feed (1,386 against 2,048 kcal), and more processing (341 against 105 kcal).

The next chart shows the chain of the world in 1968, in kilocalories.

In [11]:
# The figures for the world in 1968, in kcal per person per day.
world_1968 = energy_fbs[(energy_fbs["country"] == "World") & (energy_fbs["year"] == 1968)].iloc[0]
error = "The world's food, feed or animal products in 1968 have changed."
assert [round(world_1968[stage], -1) for stage in ["food", "feed", "animal_products"]] == [2330, 1120, 440], error
# Tourist consumption and data adjustments, which our static chart leaves out.
assert world_1968["tourist_consumption"] == 0, "The world's tourist consumption in 1968 is no longer zero."
assert round(100 * world_1968["data_adjustments"] / world_1968["food"]) == 1, "The world's data adjustments in 1968 have changed."

plot_chain(energy_fbs, country="World", year=1968, unit="kcal", title="The world's food supply chain in 1968, in kilocalories (built from FBS)")

The food supply is about 2,330 kcal per person per day.
Feed is about 1,120 kcal, and animal products are about 440 kcal.
The differences above also apply to 1968: feed excludes oilseed cakes (and grass, as in SCL), and processing includes the calories of the cakes that are fed to animals.

In our static chart, we leave out tourist consumption, which is zero for the world, and data adjustments, which are about 1% of food.

## Conclusions

FAOSTAT gives almost all figures of its food balances only in tonnes, except the food available to eat, which it also gives in calories and protein.
We convert the other figures with the density of each item: its food calories or protein divided by its food tonnes, as FAO reports them.
The result is a chain that follows each country's food from the crops harvested to the food available to eat, in calories, protein, and mass.
For the world in 2023, the crops harvested hold about 6,000 kcal per person per day, and about 3,000 kcal per person per day are available as food.

The main choices we made are:

- We build the chain from SCL instead of FBS, because SCL has individual products and the by-products of processing, such as oilseed cakes.
That way, the cakes that farm animals eat appear in feed, where they belong.
- Items that people hardly eat, such as cakes and brans, get fixed densities from FAO's nutrient conversion table or from Feedipedia.
- Crops that are mostly processed, such as sugar cane and sunflower seed, are valued by the products that factories make from them.
- Processing never adds calories or protein.
Where the data says it does, we move the gain to data adjustments.

The main limitations are:

- The chain built from SCL starts in 2010, and some countries, such as Japan, are missing, because FAO left them out of its latest release.
- Processing is the least certain stage, because it absorbs the inconsistencies of FAO's quantities and of the densities.
- Feed only includes what passes through the balance, so the grass and pasture that grazing animals eat are missing.
- Data adjustments are small for the world (1.4% of food in 2023), but larger than 10% of food in some countries.